## Continuing with Andrej Karpathy's exercises to experiment with splitting up the training set

<span style = "color:green;"> E01: train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model? </span>

<span style = "color:green;"> E02: split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see? </span>

- The loss values seem consistent with each other, I think this suggests that the dev and test sample sets are representative of the overal training set - meaning they are sufficiently large and doing contain an incredible number of outliers.

<span style = "color:green;"> E03: use the dev set to tune the strength of smoothing (or regularization) for the trigram model - i.e. try many possibilities and see which one works best based on the dev set loss. What patterns can you see in the train and dev set loss as you tune this strength? Take the best setting of the smoothing and evaluate on the test set once and at the end. How good of a loss do you achieve? </span>

<span style = "color:green;"> E04: we saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W? </span>

<span style = "color:green;"> E05: look up and use F.cross_entropy instead. You should achieve the same result. Can you think of why we'd prefer to use F.cross_entropy instead? </span>

- I get the same result. This is nice as it massively simplifies the forward pass and loss calculation functions, it is also numerically more stable and efficient and probably has GPU capabilities. 

<span style = "color:green;"> E06: meta-exercise! Think of a fun/interesting exercise and complete it. </span>

- added an autoconverging training feature + classes for different Models



**We will split the training set into 80% training, 10% development, and 10% test.**

Firstly,  we import the data in as before.

In [2]:
words = open('names.txt', 'r').read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

Similarly we import over some code that will be used to index each character and import pytorch.

In [3]:
import torch
import torch.nn.functional as F

# chars = sorted(list(set(''.join(words)))) # returns the unique characters in the list of words, sorted alphabetically
# stoi = {s: i + 1 for i, s in enumerate(chars)} # assigns a unique integer index to each character, starting from 1
# stoi['.'] = 0 # assigns the index 0 to the '.' character, which is used at the start and end of each word.
# itos = {i: s for s, i in stoi.items()} # creates a reverse mapping from integer indices back to characters.

# Defining classes

I think it makes sense to define Bigram and Trigram classes such that I can call the data handling, gradient descent, and sampling as internal functions.

In [ ]:
class BigramLanguageModel:
    def __init__(self, words, regularization_strength = 0.01):
        # creates a random number generator with a fixed seed for reproducibility (The same as used by Karpathy as a consistency check)
        g = torch.Generator().manual_seed(2147483647) 
        self.reg_strength = regularization_strength
        self.xs = []
        self.ys = []

        chars = sorted(list(set(''.join(words)))) # returns the unique characters in the list of words, sorted alphabetically
        self.stoi = {s: i + 1 for i, s in enumerate(chars)} # assigns a unique integer index to each character, starting from 1
        self.stoi['.'] = 0 # assigns the index 0 to the '.' character, which is used at the start and end of each word.
        self.itos = {i: s for s, i in self.stoi.items()} # creates a reverse mapping from integer indices back to characters.

        for w in words:
            chs = ['.'] + list(w) + ['.'] # add start and end tokens
            for ch1, ch2 in zip(chs, chs[1:]):
                self.xs.append(self.stoi[ch1]) # input character index
                self.ys.append(self.stoi[ch2]) # target character index

        self.xs = torch.tensor(self.xs) # convert to tensor
        self.ys = torch.tensor(self.ys)
  
        # Weights for the bigram model, initialized randomly.
        self.W = torch.randn((27, 27), generator=g, requires_grad=True) # 27x27 weight matrix for bigram probabilities (26 letters + '.')


    # def forward(self):
    #     # Old slower method
    #     # xenc = F.one_hot(self.xs, num_classes=27).float() # one-hot encode the input character indices
    #     # logits = xenc @ self.W # compute logits for the next character

    #     # Faster with correct matrix indexing instead - also less memory usage and more efficient.
    #     logits = self.W[self.xs]
    #     counts = logits.exp() # convert logits to counts
    #     self.probs = counts / counts.sum(1, keepdim=True) # normalize counts to get
    #     return self.probs
    
    # def loss(self):
    #     # compute the negative log likelihood loss
    #     return -self.probs[torch.arange(len(self.ys)), self.ys].log().mean() + self.reg_strength * (self.W**2).mean()

    # ========= Use of F.cross_entropy instead ================
    def forward(self): 
        self.logits = self.W[self.xs]

    def loss(self):
        return F.cross_entropy(self.logits, self.ys) + self.reg_strength * (self.W**2).mean()

    def gradient_descent(self, iterations, learning_rate=0.1, verbose=False):
        for i in range(iterations):
            self.forward() # compute probabilities
            self.W.grad = None
            loss = self.loss()
            loss.backward() # compute gradients
            self.W.data -= learning_rate * self.W.grad # gradient descent step
            
            if verbose:
                print(f"step {i}: loss = {self.loss().item():.4f}") # print the loss at each step
        return self.loss().item() # return the final loss after training
    

    def sample(self, num_samples=5):
        g = torch.Generator().manual_seed(2147483647)
        for _ in range(num_samples):
            out = []
            ix = 0  # start token '.'
            while True:
                # xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
                # logits = xenc @ self.W
                logits = self.W[ix]
                counts = logits.exp()
                probs = counts / counts.sum()
                ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
                if ix == 0:  # end token '.'
                    break
                out.append(self.itos[ix])
            print(''.join(out))

    def train(self, iterations_per_round=50, initial_lr=1.0, min_lr=0.001, decay=0.5, patience=3):
        lr = initial_lr
        best_loss = float('inf')
        rounds_since_improvement = 0
        r = 0

        while True:
            loss_val = self.gradient_descent(iterations=iterations_per_round, learning_rate=lr)
            print(f"round {r:3d}: loss = {loss_val:.4f}  lr = {lr:.5f}")
            r += 1

            if loss_val < best_loss - 1e-4:
                best_loss = loss_val
                rounds_since_improvement = 0
            else:
                rounds_since_improvement += 1

            if rounds_since_improvement >= patience:
                lr *= decay
                rounds_since_improvement = 0
                print(f"  → lr decayed to {lr:.5f}")
                if lr < min_lr:
                    print(f"Converged at loss = {best_loss:.4f}")
                    break

    def evaluate_on(self, words_ev):
        xs, ys = [], []
        for w in words_ev:
            chs = ['.'] + list(w) + ['.']
            for ch1, ch2 in zip(chs, chs[1:]):
                xs.append(self.stoi[ch1])
                ys.append(self.stoi[ch2])
        xs = torch.tensor(xs)
        ys = torch.tensor(ys)

        # xenc = F.one_hot(xs, num_classes=27).float()
        # logits = xenc @ self.W
        logits = self.W[xs]
        counts = logits.exp()
        probs = counts / counts.sum()
        return (-probs[torch.arange(len(ys)), ys].log().mean()).item()


# Suggested usage:
# bigram_model = BigramLanguageModel(words)
# bigram_model.train(iterations_per_round=100, initial_lr=100.0, min_lr=0.5, decay=0.5, patience=3)
# bigram_model.evaluate_on(words_dev)

In [13]:
class TrigramLanguageModel:
    def __init__(self, words, regularization_strength= 0.01):
        # creates a random number generator with a fixed seed for reproducibility (The same as used by Karpathy as a consistency check)
        g = torch.Generator().manual_seed(2147483647) 
        self.xs = []
        self.ys = []
        self.reg_strength = regularization_strength

        chars = sorted(list(set(''.join(words)))) # returns the unique characters in the list of words, sorted alphabetically
        self.stoi = {s: i + 1 for i, s in enumerate(chars)} # assigns a unique integer index to each character, starting from 1
        self.stoi['.'] = 0 # assigns the index 0 to the '.' character, which is used at the start and end of each word.
        self.itos = {i: s for s, i in self.stoi.items()} # creates a reverse mapping from integer indices back to characters.

        for w in words:
            chs = ['.', '.'] + list(w) + ['.'] # add start and end tokens
            for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
                self.xs.append((self.stoi[ch1], self.stoi[ch2])) # input character indices (bigram)
                self.ys.append(self.stoi[ch3]) # target character index

        self.xs = torch.tensor(self.xs) # convert to tensor
        self.ys = torch.tensor(self.ys)
  
        # Weights for the trigram model, initialized randomly.
        self.W = torch.randn((27, 27, 27), generator=g, requires_grad=True) # weight matrix for trigram probabilities (27 x 27 x 27 possible next characters)

    # def forward(self):
    #     # Old method from tutorial - inefficient and slower. One_hot method just selects a row in W.
    #     # xenc1 = F.one_hot(self.xs[:, 0], num_classes=27).float()  # one-hot encode the first character index
    #     # xenc2 = F.one_hot(self.xs[:, 1], num_classes=27).float()  # one-hot encode the second character index
    #     # logits = (xenc1 @ self.W.view(27, 27 * 27)).view(-1, 27, 27)  # compute logits for the next character
    #     # logits = (logits * xenc2.unsqueeze(2)).sum(1)             # contract with the second character encoding to get final logits

    #     #Can do this much faster with correct matrix indexing
    #     logits = self.W[self.xs[:, 0], self.xs[:, 1]]

    #     counts = logits.exp() # convert logits to counts
    #     self.probs = counts / counts.sum(1, keepdim=True) # normalize counts to get probabilities
    #     return self.probs
    
    # def loss(self):
    #     # compute the negative log likelihood loss
    #     return -self.probs[torch.arange(len(self.ys)), self.ys].log().mean() + self.reg_strength * (self.W**2).mean()

    # ======== F.cross_entropy method =============
    def forward(self):
        self.logits = self.W[self.xs[:, 0], self.xs[:, 1]]

    def loss(self):
        return F.cross_entropy(self.logits, self.ys) + self.reg_strength * (self.W**2).mean()
    

    def gradient_descent(self, iterations, learning_rate=0.1, verbose=False):
        for i in range(iterations):
            self.forward() # compute probabilities
            self.W.grad = None
            loss = self.loss()
            loss.backward()
            self.W.data -= learning_rate * self.W.grad # gradient descent step
            
            if verbose:
                print(f"step {i}: loss = {self.loss().item():.4f}") # print the loss at each step
        return self.loss().item() # return the final loss after training
    
    def sample(self, num_samples=5):
        g = torch.Generator().manual_seed(2147483647)
        for _ in range(num_samples):
            out = []
            ix1, ix2 = 0, 0  # start tokens '.'
            while True:
                logits = self.W[ix1, ix2]
                counts = logits.exp()
                probs = counts / counts.sum()
                ix3 = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
                if ix3 == 0:  # end token '.'
                    break
                out.append(self.itos[ix3])
                
                ix1, ix2 = ix2, ix3  # shift the bigram window
            print(''.join(out))

    def train(self, iterations_per_round=50, initial_lr=1.0, min_lr=0.001, decay=0.5, patience=3):
        lr = initial_lr
        best_loss = float('inf')
        rounds_since_improvement = 0
        r = 0

        while True:
            loss_val = self.gradient_descent(iterations=iterations_per_round, learning_rate=lr)
            print(f"round {r:3d}: loss = {loss_val:.4f}  lr = {lr:.5f}")
            r += 1

            if loss_val < best_loss - 1e-4:
                best_loss = loss_val
                rounds_since_improvement = 0
            else:
                rounds_since_improvement += 1

            if rounds_since_improvement >= patience:
                lr *= decay
                rounds_since_improvement = 0
                print(f"  → lr decayed to {lr:.5f}")
                if lr < min_lr:
                    print(f"Converged at loss = {best_loss:.4f}")
                    break

    def evaluate_on(self, words_ev):
            xs, ys = [], []
            for w in words_ev:
                chs = ['.', '.'] + list(w) + ['.']
                for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
                    xs.append((self.stoi[ch1], self.stoi[ch2]))
                    ys.append(self.stoi[ch3])
            xs = torch.tensor(xs)
            ys = torch.tensor(ys)

            logits = self.W[xs[:,0], xs[:, 1]]
            counts = logits.exp()
            probs = counts / counts.sum()
            return (-probs[torch.arange(len(ys)), ys].log().mean()).item()

# Suggested usage:
# trigram_model = TrigramLanguageModel(words)
# trigram_model.train(iterations_per_round=100, initial_lr=100.0, min_lr=0.5, decay=0.5, patience=3)
# trigram_model.evaulate_on(words_dev)

# Splitting the data into training/development/testing training sets

In [5]:
# We want to randomly sample the words such that
# - 80% of the words are in the training set
# - 10% in the development set
# - 10% in the test set. 

def uniform_sample(words, train_ratio=0.8, dev_ratio=0.1, test_ratio=0.1):
    assert abs(train_ratio + dev_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1"
    g = torch.Generator().manual_seed(2147483647)
    indices = torch.randperm(len(words), generator=g)
    
    train_end = int(train_ratio * len(words))
    dev_end = train_end + int(dev_ratio * len(words))
    
    words_train = [words[i] for i in indices[:train_end]]
    words_dev = [words[i] for i in indices[train_end:dev_end]]
    words_test = [words[i] for i in indices[dev_end:]]
    
    return words_train, words_dev, words_test

words_train, words_dev, words_test = uniform_sample(words)

Now we wish to train the models on the training set 'words_train'

In [14]:
# First we initialise the models.
bigramModel = BigramLanguageModel(words_train)
trigramModel = TrigramLanguageModel(words_train)

In [15]:
# Now we train, this takes a few minutes with the current code. I believe this can be substantially improved in later exercises.

bigramModel.train(iterations_per_round=10, initial_lr=100.0, min_lr=0.5)
trigramModel.train(iterations_per_round=10, initial_lr=100, min_lr=0.5)

round   0: loss = 2.6118  lr = 100.00000
round   1: loss = 2.5484  lr = 100.00000
round   2: loss = 2.5298  lr = 100.00000
round   3: loss = 2.5195  lr = 100.00000
round   4: loss = 2.5172  lr = 100.00000
round   5: loss = 2.5117  lr = 100.00000
round   6: loss = 2.5129  lr = 100.00000
round   7: loss = 2.5086  lr = 100.00000
round   8: loss = 2.5109  lr = 100.00000
round   9: loss = 2.5072  lr = 100.00000
round  10: loss = 2.5099  lr = 100.00000
round  11: loss = 2.5064  lr = 100.00000
round  12: loss = 2.5092  lr = 100.00000
round  13: loss = 2.5059  lr = 100.00000
round  14: loss = 2.5089  lr = 100.00000
round  15: loss = 2.5055  lr = 100.00000
round  16: loss = 2.5086  lr = 100.00000
round  17: loss = 2.5053  lr = 100.00000
round  18: loss = 2.5084  lr = 100.00000
round  19: loss = 2.5052  lr = 100.00000
round  20: loss = 2.5083  lr = 100.00000
round  21: loss = 2.5050  lr = 100.00000
round  22: loss = 2.5082  lr = 100.00000
round  23: loss = 2.5049  lr = 100.00000
round  24: loss 

Now we sample from the dev and test sets and compare.

In [22]:
# Bigram Model
dev_loss_bigram = bigramModel.evaluate_on(words_dev)
test_loss_bigram = bigramModel.evaluate_on(words_test)

# Trigram Model
dev_loss_trigram = trigramModel.evaluate_on(words_dev)
test_loss_trigram = trigramModel.evaluate_on(words_test)

print(f"Bigram Model loss evaulated on: dev set = {dev_loss_bigram} and test set = {test_loss_bigram}")
print(f"Trigram Model loss evaluated on: dev set = {dev_loss_trigram} and test set = {test_loss_trigram}")

Bigram Model loss evaulated on: dev set = 2.4603891372680664 and test set = 2.4610395431518555
Trigram Model loss evaluated on: dev set = 2.2344162464141846 and test set = 2.2254371643066406


# Exploring Regularization constant values effect on the evaluated loss.

The loss on the two sets seem very similar to each other and to the training set. I guess this is to be expected as the dev and test sets are sufficiently large to fairly represent the training set. If the sets were smaller and contained many statistical outliers, I would expect a higher loss function using a model trained on the large training set. 

Now I will retrain the models with varying regularisation constants, and see how this effects the loss evaluated on the dev set. 

In [ ]:
regularisation_values = [ 0, 0.005, 0.01, 0.1, 0.3, 0.5, 0.7, 1.0, 3.0, 1e-4]
dev_loss_bigram_reg = []
dev_loss_trigram_reg = []

for rv in regularisation_values:
    bigramModel = BigramLanguageModel(words_train, regularization_strength=rv)
    trigramModel = TrigramLanguageModel(words_train, regularization_strength=rv)

    bigramModel.train(iterations_per_round=10, initial_lr=100, min_lr=0.01)
    trigramModel.train(iterations_per_round=10, initial_lr=100, min_lr=0.01)

    dev_loss_bigram_reg.append(bigramModel.evaluate_on(words_dev))
    dev_loss_trigram_reg.append(trigramModel.evaluate_on(words_dev))



round   0: loss = 2.6013  lr = 100.00000
round   1: loss = 2.5368  lr = 100.00000
round   2: loss = 2.5145  lr = 100.00000
round   3: loss = 2.5050  lr = 100.00000
round   4: loss = 2.4985  lr = 100.00000
round   5: loss = 2.4959  lr = 100.00000
round   6: loss = 2.4918  lr = 100.00000
round   7: loss = 2.4921  lr = 100.00000
round   8: loss = 2.4880  lr = 100.00000
round   9: loss = 2.4900  lr = 100.00000
round  10: loss = 2.4859  lr = 100.00000
round  11: loss = 2.4885  lr = 100.00000
round  12: loss = 2.4847  lr = 100.00000
round  13: loss = 2.4874  lr = 100.00000
round  14: loss = 2.4837  lr = 100.00000
round  15: loss = 2.4866  lr = 100.00000
round  16: loss = 2.4830  lr = 100.00000
round  17: loss = 2.4860  lr = 100.00000
round  18: loss = 2.4825  lr = 100.00000
round  19: loss = 2.4855  lr = 100.00000
round  20: loss = 2.4821  lr = 100.00000
round  21: loss = 2.4851  lr = 100.00000
round  22: loss = 2.4817  lr = 100.00000
round  23: loss = 2.4848  lr = 100.00000
round  24: loss 

In [27]:
print(dev_loss_bigram_reg)
print(dev_loss_trigram_reg)

[2.455008029937744, 2.4576256275177, 2.460387945175171, 2.5035312175750732, 2.572479486465454, 2.624981164932251, 2.6683802604675293, 2.7223148345947266, 2.9259870052337646, 2.455038070678711]
[2.233438491821289, 2.233853578567505, 2.2344398498535156, 2.255481243133545, 2.299704074859619, 2.334963083267212, 2.364901065826416, 2.403048276901245, 2.5640552043914795, 2.233440399169922]


Found that 1e-4 was the best regularization for the Trigram, and no reg was best for the Bigram. I should perform a more detailed fitting of this value around smaller regularization values. But for the purposes of learning how the code works, I don't think it is beneficial as it takes longer than I would like. 

In [ ]:
rv_b, rv_t = 0, 1e-4

bigramModel = BigramLanguageModel(words_train, regularization_strength=rv_b)
trigramModel = TrigramLanguageModel(words_train, regularization_strength=rv_t)

bigramModel.train(iterations_per_round=10, initial_lr=100, min_lr=0.01)
trigramModel.train(iterations_per_round=10, initial_lr=100, min_lr=0.01)

test_loss_b = bigramModel.evaluate_on(words_test)
test_loss_t = trigramModel.evaluate_on(words_test)

In [31]:
print(f"Retrained Bigram model evalauted on the test set gives a loss of: {test_loss_b} ")
print(f"Retrained Trigram model evaluated on the test set gives a loss of: {test_loss_t}")

Retrained Bigram model evalauted on the test set gives a loss of: 2.456041097640991 
Retrained Trigram model evaluated on the test set gives a loss of: 2.224414587020874


We find a slight improvement to to trigram model now we have found a more optimal regularization factor. I reiterate, this is not the ideal solution at all, just getting a small bit of practice with changing parameters to minimise the loss function on a development sample before applying to the final test set. 